Imports & Setup

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [5]:
PALETTE = {
    'bg':        '#0D0F1A',
    'card':      '#131625',
    'accent1':   '#6C63FF',   # violet
    'accent2':   '#FF6584',   # coral
    'accent3':   '#43E97B',   # mint
    'accent4':   '#FFD166',   # amber
    'text':      '#E8E8F0',
    'subtext':   '#7C7CA0',
    'churn':     '#FF6584',
    'retain':    '#43E97B',
    'grid':      '#1E2035',
}

In [6]:
plt.rcParams.update({
    'figure.facecolor':  PALETTE['bg'],
    'axes.facecolor':    PALETTE['card'],
    'axes.edgecolor':    PALETTE['grid'],
    'axes.labelcolor':   PALETTE['text'],
    'axes.titlecolor':   PALETTE['text'],
    'xtick.color':       PALETTE['subtext'],
    'ytick.color':       PALETTE['subtext'],
    'grid.color':        PALETTE['grid'],
    'grid.linewidth':    0.6,
    'text.color':        PALETTE['text'],
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    14,
    'axes.titleweight':  'bold',
    'axes.labelsize':    11,
    'legend.facecolor':  PALETTE['card'],
    'legend.edgecolor':  PALETTE['grid'],
    'legend.labelcolor': PALETTE['text'],
    'figure.dpi':        130,
})

In [7]:
print("Libraries loaded | Style configured")

Libraries loaded | Style configured


Load Dataset (CSV)


In [9]:
file_path = 'synthetic_churn_data.csv'   # make sure file is in same folder

df = pd.read_csv(file_path)

# Convert signup_date to datetime
df['signup_date'] = pd.to_datetime(df['signup_date'])

# Quick sanity check
print(f"Dataset loaded successfully | Shape: {df.shape}")
print(f"Churn Rate: {df['churn'].mean()*100:.2f}%")

df.head()

Dataset loaded successfully | Shape: (8000, 16)
Churn Rate: 55.86%


,customer_id,gender,age,country,signup_date,plan_type,contract_type,monthly_fee,payment_method,login_frequency_per_week,avg_session_minutes,features_used_count,last_login_days_ago,support_tickets,avg_response_time_hours,churn
0,aec40fea-71fa-4606-a202-c4c7942aa51e,Other,25,Pakistan,2023-01-31,Basic,Monthly,8.49,Card,10,95,18,5,9,20.8,1
1,e48046ae-ccba-4836-97ca-6f65688863bd,Male,32,UAE,2020-04-18,Premium,Monthly,130.41,PayPal,3,58,19,17,0,36.7,1
2,413bc3b5-ec0a-4425-84ac-528a8d999411,Female,39,USA,2021-09-28,Basic,Yearly,7.56,PayPal,1,46,12,38,4,38.9,1
3,d7e5adb8-9066-44da-9984-ba66cc75b485,Other,25,UK,2020-11-18,Premium,Yearly,138.23,JazzCash,9,25,3,2,10,11.7,0
4,5f668e08-dc1c-498d-a5b5-70fb6592ffe5,Male,32,Canada,2021-02-17,Standard,Yearly,53.22,JazzCash,2,48,12,13,10,13.5,0


Exploratory Data Analysis (EDA)


In [10]:
print("=" * 55)
print("  DATASET OVERVIEW")
print("=" * 55)
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nChurn distribution:")
print(df['churn'].value_counts())
print(f"\nChurn Rate: {df['churn'].mean()*100:.2f}%")
print("\nNumerical summary:")
df.describe().round(2)

  DATASET OVERVIEW
customer_id                         object
gender                              object
age                                  int64
country                             object
signup_date                 datetime64[ns]
plan_type                           object
contract_type                       object
monthly_fee                        float64
payment_method                      object
login_frequency_per_week             int64
avg_session_minutes                  int64
features_used_count                  int64
last_login_days_ago                  int64
support_tickets                      int64
avg_response_time_hours            float64
churn                                int64
dtype: object

Missing values:
customer_id                 0
gender                      0
age                         0
country                     0
signup_date                 0
plan_type                   0
contract_type               0
monthly_fee                 0
payment_method        

,age,signup_date,monthly_fee,login_frequency_per_week,avg_session_minutes,features_used_count,last_login_days_ago,support_tickets,avg_response_time_hours,churn
count,8000.00,8000,8000.00,8000.00,8000.00,8000.00,8000.00,8000.00,8000.00,8000.00
mean,44.15,2023-01-05 21:54:32.400000256,61.84,7.00,60.40,10.48,29.75,4.97,24.50,0.56
min,18.00,2020-01-01 00:00:00,5.01,0.00,1.00,1.00,0.00,0.00,1.00,0.00
25%,31.00,2021-07-07 18:00:00,23.42,3.00,31.00,5.00,14.00,2.00,12.90,0.00
50%,44.00,2023-01-15 00:00:00,53.34,7.00,60.00,11.00,30.00,5.00,24.60,1.00
75%,58.00,2024-07-05 00:00:00,96.69,11.00,90.00,15.00,45.00,8.00,36.00,1.00
max,70.00,2025-12-31 00:00:00,150.00,14.00,120.00,20.00,60.00,10.00,48.00,1.00
std,15.42,NaN,42.67,4.33,34.34,5.74,17.73,3.16,13.49,0.50
